<a href="https://colab.research.google.com/github/MorganJhn/jupyterlab/blob/main/Jupyterlab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Install JupyterLab, collaboration tools, and cloudflared
!pip install -q jupyterlab jupyter-collaboration
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm cloudflared-linux-amd64.deb
print("Setup completed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.3 MB/s eta 0:00:00
Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Setup completed successfully!


In [2]:
%%bash
# Step 2: Start JupyterLab in the background
nohup jupyter lab \
  --ip=0.0.0.0 \
  --port=8888 \
  --no-browser \
  --ServerApp.token='your_secure_token' \
  --ServerApp.allow_origin='*' > jupyter.log 2>&1 &

In [3]:
# Step 3: Start cloudflared background tunnel and retrieve the live URL
!nohup cloudflared tunnel --url http://localhost:8888 > cloudflared.log 2>&1 &

import time
import re

print("Waiting for Cloudflare Tunnel to spin up...")
time.sleep(8)

try:
    with open("cloudflared.log", "r") as f:
        log_content = f.read()
    urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
    if urls:
        print("\n=========================================================")
        print(f"Your secure JupyterLab URL: {urls[0]}")
        print("Token: your_secure_token")
        print("=========================================================")
    else:
        print("Tunnel URL not found yet. Run this cell again or check logs:")
        print(log_content[-500:])
except Exception as e:
    print(f"Error reading logs: {e}")

Waiting for Cloudflare Tunnel to spin up...

Your secure JupyterLab URL: https://pts-offering-patricia-coral.trycloudflare.com
Token: your_secure_token


In [4]:
# Optional: View active background processes
!ps aux | grep -E 'jupyter|cloudflared'

root         106  1.4  1.2 629348 164008 ?       Sl   09:21   0:08 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        2312 12.4  0.8 663524 115244 ?       Ssl  09:30   0:04 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-93de3246-5abb-4982-bee6-61d9e9f68b07.json
root        2997 74.3  1.2 267284 167448 ?       Rl   09:31   0:06 /usr/bin/python3 /usr/local/bin/jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --ServerApp.token=your_secure_token --ServerApp.allow_origin=*
root        2999  4.4  0.2 1294484 38816 ?       Sl   09:31   0:00 cloudflared tunnel --url http://localhost:8888
root        3075  0.0  0.0   7344  3636 ?        S    09:31   0:00 /bin/bash -c ps au